## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)

In [1]:
import polars as pl
import rmm

In [5]:
mr = rmm.mr.PrefetchResourceAdaptor(rmm.mr.PoolMemoryResource(rmm.mr.ManagedMemoryResource()))
engine = pl.GPUEngine(device=0, memory_resource=mr, raise_on_fail=False) # Can configure engine to run on CPU if fails

In [6]:
zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}


df_zones = pl.scan_csv("../data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

# Taxi operation efficiency
df = (
    pl.scan_parquet("../data/nyc-taxi/green.parquet")
    .with_columns(
        ((pl.col("tpep_dropoff_datetime") - pl.col("tpep_pickup_datetime")).dt.total_minutes()).alias(
            "trip_duration_minutes"
        )
    )
    .filter((pl.col("trip_duration_minutes") > 0) & (pl.col("trip_distance") > 0) & (pl.col("fare_amount") > 0))
    .with_columns(
        [
            (pl.col("fare_amount") / pl.col("trip_duration_minutes")).alias("revenue_per_minute"),
            (pl.col("fare_amount") / pl.col("trip_distance")).alias("revenue_per_mile"),
            (pl.col("total_amount") / pl.col("trip_duration_minutes")).alias("total_revenue_per_minute"),
            (pl.col("trip_distance") / (pl.col("trip_duration_minutes") / 60)).alias("speed_mph"),
            (pl.col("fare_amount") * pl.col("trip_distance") / pl.col("trip_duration_minutes")).alias(
                "efficiency_score"
            ),
        ]
    )
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [7]:
df.group_by("PU_Borough").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).with_columns((pl.col("trip_count") / 8 * 100).alias("percentage_of_trips")).sort(
    "trip_count", descending=True
).collect(engine=engine)

PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip,percentage_of_trips
str,u32,f32,f32,f32,f32,f64
"""Brooklyn""",27966145,3.6394912e8,13.013918,6.019091,1.559054,3.4958e8
"""Manhattan""",25638292,2.83141024e8,11.043677,4.674914,1.151222,3.2047865e8
"""Queens""",22754192,2.78632992e8,12.245348,5.599979,0.881084,2.844274e8
"""Bronx""",4773827,7.1688056e7,15.016894,16.331867,0.468896,5.9673e7
"""Unknown""",51708,672063.875,12.99729,4.074217,1.358674,646350.0
"""N/A""",24260,988989.25,40.766251,25.785395,2.914987,303250.0
"""Staten Island""",17307,690061.5,39.871815,25.785686,1.48944,216337.5
"""EWR""",269,16617.927734,61.776684,12.505535,8.100408,3362.5


In [ ]:
# df.collect(engine=engine)

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,extra,mta_tax,tip_amount,tolls_amount,airport_fee,improvement_surcharge,total_amount,payment_type,congestion_surcharge,trip_duration_minutes,revenue_per_minute,revenue_per_mile,total_revenue_per_minute,speed_mph,efficiency_score
i16,datetime[ms],datetime[ms],str,u8,u16,u16,u16,f32,f32,f32,f32,f32,f32,f32,f32,f32,u8,f32,i64,f64,f32,f64,f64,f64
2,2014-01-01 00:17:26,2014-01-01 00:37:11,"""N""",1,17,225,1,2.28,13.5,0.5,0.5,0.0,0.0,null,null,14.5,2,null,19,0.710526,5.921053,0.763158,7.2,1.62
1,2014-01-01 00:29:12,2014-01-01 00:37:43,"""N""",1,127,241,1,2.1,9.0,0.0,0.5,0.0,0.0,null,null,9.5,2,null,8,1.125,4.285715,1.1875,15.749999,2.3625
2,2014-01-01 00:31:35,2014-01-01 00:44:09,"""N""",1,166,243,1,4.72,15.5,0.5,0.5,4.0,0.0,null,null,20.5,1,null,12,1.291667,3.283898,1.708333,23.599999,6.096666
2,2014-01-01 00:07:01,2014-01-01 00:21:54,"""N""",1,7,157,1,2.88,13.0,0.5,0.5,2.88,0.0,null,null,16.879999,1,null,14,0.928571,4.513889,1.205714,12.342858,2.674286
2,2014-01-01 00:26:43,2014-01-01 00:37:17,"""N""",1,83,197,2,3.8,13.0,0.5,0.5,0.0,0.0,null,null,14.0,2,null,10,1.3,3.421053,1.4,22.8,4.94
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2,2024-12-31 22:24:00,2024-12-31 22:52:00,null,null,7,148,null,8.96,41.27,0.0,0.5,0.0,0.0,null,1.0,45.52,null,null,28,1.473929,4.606027,1.625714,19.2,13.2064
2,2024-12-31 23:42:09,2025-01-01 00:07:20,null,null,256,107,null,6.27,29.299999,0.0,0.5,3.24,6.94,null,1.0,43.73,null,null,25,1.172,4.673046,1.7492,15.048,7.34844
2,2024-12-31 23:23:00,2024-12-31 23:37:00,null,null,42,140,null,4.82,22.74,0.0,0.5,5.4,0.0,null,1.0,32.389999,null,null,14,1.624286,4.717842,2.313571,20.657144,7.829057


## UVM for single GPU
[UVM, data larger than GPU memory post](https://developer.nvidia.com/blog/how-to-work-with-data-exceeding-vram-in-the-polars-gpu-engine/)


## GPU Streaming

In [1]:
## GPU Streaming